In [1]:
"""
Quantum-Assisted Irrigation Water Allocation Using QUBO + QAOA
Hackathon project

Project flow:
Irrigation data
    -> QUBO
    -> Ising Hamiltonian
    -> QAOA circuit
    -> measured allocation
    -> quantum threshold comparator
    -> feasibility decision
    -> fault-tolerant research extension

Test instance:
Crop A: water=1, benefit=3
Crop B: water=2, benefit=5
Crop C: water=3, benefit=6
Available water = 3

Expected optimum:
A + B -> water 3 -> benefit 8
bit decision x_A x_B x_C = 110
"""

import itertools
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Optional simulator/plot imports
try:
    from qiskit_aer import Aer
    HAVE_AER = True
except ImportError:
    HAVE_AER = False

try:
    from qiskit.visualization import plot_histogram
    HAVE_PLOT = True
except ImportError:
    HAVE_PLOT = False

from scipy.optimize import minimize


# ============================================================
# STEP 1 - IRRIGATION PROBLEM
# ============================================================

CROPS = ["A", "B", "C"]

WATER = {
    "A": 1,
    "B": 2,
    "C": 3,
}

BENEFIT = {
    "A": 3,
    "B": 5,
    "C": 6,
}

AVAILABLE_WATER = 3


def evaluate_allocation(x):
    """Return water used and benefit for x=(xA,xB,xC)."""
    water_used = sum(WATER[CROPS[i]] * x[i] for i in range(3))
    benefit = sum(BENEFIT[CROPS[i]] * x[i] for i in range(3))
    return water_used, benefit


def classical_benchmark():
    """Exhaustively solve the tiny irrigation instance."""
    best_x = None
    best_benefit = -1

    rows = []

    for x in itertools.product([0, 1], repeat=3):
        water_used, benefit = evaluate_allocation(x)
        feasible = water_used <= AVAILABLE_WATER

        rows.append((x, water_used, benefit, feasible))

        if feasible and benefit > best_benefit:
            best_benefit = benefit
            best_x = x

    return best_x, best_benefit, rows


# ============================================================
# STEP 1B - QUBO
# ============================================================

# Penalty formulation:
#
# C(x) = -benefit(x) + P*(water(x)-3)^2
#
# For this small instance P=10.
#
# Expanded:
# C = -53*xA -85*xB -96*xC
#     +40*xA*xB +60*xA*xC +120*xB*xC +90

PENALTY = 10.0

LINEAR = np.array([-53.0, -85.0, -96.0])
QUADRATIC = {
    (0, 1): 40.0,
    (0, 2): 60.0,
    (1, 2): 120.0,
}
QUBO_CONSTANT = 90.0


def qubo_energy(x):
    """Evaluate the expanded QUBO energy."""
    x = np.asarray(x, dtype=float)

    energy = QUBO_CONSTANT
    energy += np.dot(LINEAR, x)

    for (i, j), value in QUADRATIC.items():
        energy += value * x[i] * x[j]

    return float(energy)


def verify_qubo():
    """Print all QUBO energies and the minimum."""
    data = []

    for x in itertools.product([0, 1], repeat=3):
        data.append((x, qubo_energy(x)))

    data.sort(key=lambda z: z[1])

    print("\nQUBO energies (lowest first):")
    for x, e in data:
        print(f"{x}: {e:.3f}")

    print("\nQUBO optimum:", data[0][0], "energy =", data[0][1])


# ============================================================
# STEP 2 - QUBO -> ISING
# ============================================================

# Mapping x_i = (1-Z_i)/2 gives:
#
# H = offset + h_i Z_i + J_ij Z_i Z_j
#
# h = [38.5, 37.5, 36.0]
# J01=10, J02=15, J12=30
# offset=11
#
# The constant offset does not affect the QAOA optimum.

H_LINEAR = np.array([38.5, 37.5, 36.0])
H_QUADRATIC = {
    (0, 1): 10.0,
    (0, 2): 15.0,
    (1, 2): 30.0,
}
ISING_OFFSET = 11.0


# ============================================================
# STEP 2 - QAOA CIRCUIT
# ============================================================

def create_qaoa_circuit(gammas, betas, p=None, measure=False):
    """
    Build a p-layer QAOA circuit.

    Qubit mapping:
        q0 -> Crop A
        q1 -> Crop B
        q2 -> Crop C

    The displayed bit string is CBA in Qiskit's conventional
    little-endian display. The helper decode_bitstring() converts
    it back to ABC.
    """
    gammas = np.atleast_1d(gammas)
    betas = np.atleast_1d(betas)

    if p is None:
        p = len(gammas)

    if len(gammas) != p or len(betas) != p:
        raise ValueError("gammas and betas must both contain p values")

    qc = QuantumCircuit(3)

    # |+++>
    qc.h(range(3))

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        # exp(-i gamma * h_i Z_i)
        for q in range(3):
            qc.rz(2 * gamma * H_LINEAR[q], q)

        # exp(-i gamma * J_ij Z_i Z_j)
        for (i, j), J in H_QUADRATIC.items():
            qc.rzz(2 * gamma * J, i, j)

        # Mixer: exp(-i beta X)
        for q in range(3):
            qc.rx(2 * beta, q)

    if measure:
        qc.measure_all()

    return qc


def qaoa_expectation(params, p=1):
    """Expectation value of the Ising Hamiltonian using Statevector."""
    gammas = params[:p]
    betas = params[p:]

    qc = create_qaoa_circuit(gammas, betas, p=p, measure=False)
    state = Statevector.from_instruction(qc)

    probabilities = state.probabilities()

    expectation = 0.0

    # Statevector index uses q0 as the least-significant bit.
    for index, probability in enumerate(probabilities):
        if probability < 1e-14:
            continue

        # z=+1 for computational |0>, z=-1 for |1>
        z = []
        for q in range(3):
            bit = (index >> q) & 1
            z.append(1 if bit == 0 else -1)

        energy = ISING_OFFSET
        energy += sum(H_LINEAR[q] * z[q] for q in range(3))

        for (i, j), J in H_QUADRATIC.items():
            energy += J * z[i] * z[j]

        expectation += probability * energy

    return float(np.real(expectation))


def optimize_qaoa(p=1, maxiter=100):
    """Classically optimize the QAOA angles."""
    rng = np.random.default_rng(7)

    # A small random start helps avoid an unlucky deterministic start.
    x0 = rng.uniform(0, np.pi, 2 * p)

    result = minimize(
        lambda params: qaoa_expectation(params, p=p),
        x0=x0,
        method="COBYLA",
        options={"maxiter": maxiter},
    )

    return result


# ============================================================
# STEP 2/4 - MEASUREMENT
# ============================================================

def run_aer(gammas, betas, shots=4096):
    """Run the optimized circuit on Aer."""
    if not HAVE_AER:
        raise RuntimeError(
            "qiskit-aer is not installed. Run: pip install qiskit-aer"
        )

    qc = create_qaoa_circuit(gammas, betas, p=len(gammas), measure=True)

    backend = Aer.get_backend("aer_simulator")
    job = backend.run(qc, shots=shots)
    return job.result().get_counts()


def run_statevector_probabilities(gammas, betas):
    """Return probabilities without requiring Aer."""
    qc = create_qaoa_circuit(gammas, betas, p=len(gammas), measure=False)
    state = Statevector.from_instruction(qc)
    probabilities = state.probabilities()

    result = {}
    for index, probability in enumerate(probabilities):
        if probability < 1e-10:
            continue

        # Convert index -> ABC decision string.
        bits = "".join(
            str((index >> q) & 1)
            for q in range(3)
        )
        result[bits] = float(probability)

    return dict(sorted(result.items(), key=lambda kv: kv[1], reverse=True))


def decode_bitstring(bitstring):
    """
    Convert a Qiskit displayed measurement string CBA
    into the application order ABC.
    """
    cleaned = bitstring.replace(" ", "")
    if len(cleaned) < 3:
        raise ValueError("Expected a 3-bit result.")
    return cleaned[-1] + cleaned[-2] + cleaned[-3]


def report_measurements(counts):
    """Print measured allocations in application order ABC."""
    print("\nMeasured solutions:")
    for displayed, count in sorted(
        counts.items(), key=lambda kv: kv[1], reverse=True
    ):
        abc = decode_bitstring(displayed)
        x = tuple(int(v) for v in abc)
        water_used, benefit = evaluate_allocation(x)
        print(
            f"displayed={displayed:>3} -> ABC={abc}, "
            f"shots={count:>5}, water={water_used}, "
            f"benefit={benefit}, "
            f"feasible={water_used <= AVAILABLE_WATER}"
        )


# ============================================================
# STEP 3 - QUANTUM THRESHOLD COMPARATOR
# ============================================================

def threshold_comparator_2bit(threshold=2, measure=True):
    """
    Small reversible 2-bit water-threshold comparator.

    Computes:
        out <- out XOR [W >= threshold]

    Inputs:
        q0 = W0 (LSB)
        q1 = W1 (MSB)
        q2 = output

    For threshold=2:
        W=00 -> 0
        W=01 -> 0
        W=10 -> 1
        W=11 -> 1

    This simplified threshold instance is used for the hackathon.
    The user's research comparator can replace this block in the
    fault-tolerant extension.
    """
    if threshold not in (0, 1, 2, 3):
        raise ValueError("threshold must be 0, 1, 2, or 3")

    qc = QuantumCircuit(3, 1)

    # Truth-table implementation using MCX/X.
    # We flip q2 for each W value satisfying W >= threshold.
    for W in range(4):
        if W >= threshold:
            bits = [(W >> 0) & 1, (W >> 1) & 1]

            # Convert required zero controls to one controls.
            for q, value in enumerate(bits):
                if value == 0:
                    qc.x(q)

            if len([0, 1]) == 2:
                qc.mcx([0, 1], 2)

            for q, value in enumerate(bits):
                if value == 0:
                    qc.x(q)

    if measure:
        qc.measure(2, 0)

    return qc


def test_comparator_cases():
    """
    Demonstration cases are evaluated logically and the circuit
    is shown. For a live demo, prepare each W basis state and run
    the circuit.
    """
    threshold = 2
    print("\nComparator: W >= 2")
    for W in range(4):
        result = int(W >= threshold)
        print(f"W={W:02b} ({W}) -> comparator output={result}")

    qc = threshold_comparator_2bit(threshold=2, measure=True)
    print("\nComparator circuit:")
    print(qc.draw("text"))


# ============================================================
# STEP 4 - APPLICATION REPORT
# ============================================================

def print_project_summary(best_x, best_benefit, qaoa_result=None):
    water_used, benefit = evaluate_allocation(best_x)

    print("\n" + "=" * 65)
    print("FINAL IRRIGATION OPTIMIZATION SUMMARY")
    print("=" * 65)
    print(f"Classical optimum (ABC): {best_x}")
    print(f"Water used:              {water_used}")
    print(f"Benefit:                 {benefit}")
    print("Selected crops:          " +
          ", ".join(CROPS[i] for i, v in enumerate(best_x) if v))

    if qaoa_result is not None:
        print(f"QAOA optimized objective: {qaoa_result.fun:.6f}")

    print("\nInterpretation:")
    print("A=1, B=1, C=0 means irrigate Crops A and B.")
    print("The plan uses all 3 available water units and gives benefit 8.")
    print("=" * 65)


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("\nQUANTUM-ASSISTED IRRIGATION WATER ALLOCATION")
    print("============================================")

    # Step 1: classical benchmark
    best_x, best_benefit, rows = classical_benchmark()

    print("\nClassical benchmark:")
    print("Best allocation:", best_x)
    print("Best benefit:", best_benefit)
    print("Water used:", evaluate_allocation(best_x)[0])

    # QUBO verification
    verify_qubo()

    # Step 2: QAOA
    print("\nOptimizing QAOA...")
    qaoa_result = optimize_qaoa(p=1, maxiter=100)

    gamma_opt = qaoa_result.x[:1]
    beta_opt = qaoa_result.x[1:]

    print("Optimal gamma:", gamma_opt)
    print("Optimal beta :", beta_opt)
    print("QAOA expectation:", qaoa_result.fun)

    print("\nQAOA circuit:")
    print(create_qaoa_circuit(
        gamma_opt,
        beta_opt,
        p=1,
        measure=True
    ).draw("text"))

    # Statevector result works even without Aer
    probs = run_statevector_probabilities(gamma_opt, beta_opt)

    print("\nTop statevector probabilities (ABC order):")
    for bitstring, probability in list(probs.items())[:8]:
        x = tuple(int(v) for v in bitstring)
        water_used, benefit = evaluate_allocation(x)
        print(
            f"ABC={bitstring}  probability={probability:.4f}  "
            f"water={water_used}  benefit={benefit}"
        )

    # Optional Aer result
    if HAVE_AER:
        counts = run_aer(gamma_opt, beta_opt, shots=4096)
        report_measurements(counts)

        if HAVE_PLOT:
            try:
                import matplotlib.pyplot as plt
                fig = plot_histogram(counts)
                fig.savefig("qaoa_histogram.png", dpi=300, bbox_inches="tight")
                plt.close(fig)
                print("\nSaved qaoa_histogram.png")
            except Exception as exc:
                print("Histogram could not be saved:", exc)
    else:
        print(
            "\nAer is not installed. Statevector results are available. "
            "Install qiskit-aer for shot-based results."
        )

    # Step 3: comparator
    test_comparator_cases()

    # Step 4: final application statement
    print_project_summary(best_x, best_benefit, qaoa_result)



QUANTUM-ASSISTED IRRIGATION WATER ALLOCATION

Classical benchmark:
Best allocation: (1, 1, 0)
Best benefit: 8
Water used: 3

QUBO energies (lowest first):
(1, 1, 0): -8.000
(0, 0, 1): -6.000
(1, 0, 1): 1.000
(0, 1, 0): 5.000
(0, 1, 1): 29.000
(1, 0, 0): 37.000
(1, 1, 1): 76.000
(0, 0, 0): 90.000

QUBO optimum: (1, 1, 0) energy = -8.0

Optimizing QAOA...
Optimal gamma: [2.84347142]
Optimal beta : [2.75285083]
QAOA expectation: -30.69933793480566

QAOA circuit:
        ┌───┐┌────────────┐                          ┌────────────┐»
   q_0: ┤ H ├┤ Rz(218.95) ├─■────────────■───────────┤ Rx(5.5057) ├»
        ├───┤├────────────┤ │ZZ(56.869)  │           └────────────┘»
   q_1: ┤ H ├┤ Rz(213.26) ├─■────────────┼────────────■────────────»
        ├───┤├────────────┤              │ZZ(85.304)  │ZZ(170.61)  »
   q_2: ┤ H ├┤ Rz(204.73) ├──────────────■────────────■────────────»
        └───┘└────────────┘                                        »
meas: 3/════════════════════════════════════════════


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Mamtha\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Mamtha\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Mamtha\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Histogram could not be saved: numpy.core.multiarray failed to import

Comparator: W >= 2
W=00 (0) -> comparator output=0
W=01 (1) -> comparator output=0
W=10 (2) -> comparator output=1
W=11 (3) -> comparator output=1

Comparator circuit:
     ┌───┐     ┌───┐        
q_0: ┤ X ├──■──┤ X ├──■─────
     └───┘  │  └───┘  │     
q_1: ───────■─────────■─────
          ┌─┴─┐     ┌─┴─┐┌─┐
q_2: ─────┤ X ├─────┤ X ├┤M├
          └───┘     └───┘└╥┘
c: 1/═════════════════════╩═
                          0 

FINAL IRRIGATION OPTIMIZATION SUMMARY
Classical optimum (ABC): (1, 1, 0)
Water used:              3
Benefit:                 8
Selected crops:          A, B
QAOA optimized objective: -30.699338

Interpretation:
A=1, B=1, C=0 means irrigate Crops A and B.
The plan uses all 3 available water units and gives benefit 8.


In [3]:
"""Corrected 3-qubit QUBO/QAOA irrigation demo.
A=(water 1, benefit 3), B=(2,5), C=(3,6), capacity=3.
NOTE: This compact penalty model penalizes exact fill (water=3), not a general <= constraint.
It gives the intended optimum for this specific example. General inequality needs slack qubits.
"""
import itertools
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

try:
    from qiskit_aer import Aer
    HAVE_AER = True
except ImportError:
    HAVE_AER = False

WATER = np.array([1, 2, 3])
BENEFIT = np.array([3, 5, 6])
CAPACITY = 3

# QUBO: C = -53*xA -85*xB -96*xC +40*xA*xB +60*xA*xC +120*xB*xC +90
LINEAR = np.array([-53., -85., -96.])
QUADRATIC = {(0, 1): 40., (0, 2): 60., (1, 2): 120.}
QUBO_OFFSET = 90.

# Correct mapping x_i=(1-Z_i)/2:
# H = 1.5 ZA + 2.5 ZB + 3 ZC + 10 ZA ZB + 15 ZA ZC + 30 ZB ZC + 28
H_LINEAR = np.array([1.5, 2.5, 3.0])
H_QUADRATIC = {(0, 1): 10., (0, 2): 15., (1, 2): 30.}
ISING_OFFSET = 28.

def allocation(x):
    x = np.asarray(x, dtype=int)
    return int(WATER @ x), int(BENEFIT @ x)

def qubo_energy(x):
    x = np.asarray(x, dtype=float)
    e = QUBO_OFFSET + float(LINEAR @ x)
    for (i, j), b in QUADRATIC.items():
        e += b * x[i] * x[j]
    return float(e)

def ising_energy(x):
    z = 1.0 - 2.0 * np.asarray(x, dtype=float)
    e = ISING_OFFSET + float(H_LINEAR @ z)
    for (i, j), b in H_QUADRATIC.items():
        e += b * z[i] * z[j]
    return float(e)

def validate_mapping():
    for x in itertools.product([0, 1], repeat=3):
        assert np.isclose(qubo_energy(x), ising_energy(x)), (
            f"Mapping error for {x}: QUBO={qubo_energy(x)}, Ising={ising_energy(x)}"
        )
    print("PASS: QUBO and Ising energies agree for all 8 bit strings.")

def qaoa_circuit(gammas, betas, measure=False):
    gammas, betas = np.atleast_1d(gammas), np.atleast_1d(betas)
    if len(gammas) != len(betas):
        raise ValueError("gamma and beta arrays must have equal lengths")
    qc = QuantumCircuit(3)
    qc.h(range(3))
    for gamma, beta in zip(gammas, betas):
        for q in range(3):
            qc.rz(2 * gamma * H_LINEAR[q], q)
        for (i, j), b in H_QUADRATIC.items():
            qc.rzz(2 * gamma * b, i, j)
        for q in range(3):
            qc.rx(2 * beta, q)
    if measure:
        qc.measure_all()
    return qc

def expectation(params, p=1):
    qc = qaoa_circuit(params[:p], params[p:])
    probs = Statevector.from_instruction(qc).probabilities()
    e = 0.0
    for index, prob in enumerate(probs):
        x = [(index >> q) & 1 for q in range(3)]  # q0=A, q1=B, q2=C
        e += prob * ising_energy(x)
    return float(np.real(e))

def optimize_qaoa(p=1, starts=10, maxiter=300, seed=7):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(starts):
        x0 = rng.uniform(0, 2*np.pi, 2*p)
        res = minimize(lambda v: expectation(v, p), x0, method="COBYLA",
                       options={"maxiter": maxiter, "rhobeg": 0.5})
        if best is None or res.fun < best.fun:
            best = res
    return best

def statevector_results(gammas, betas):
    qc = qaoa_circuit(gammas, betas)
    probs = Statevector.from_instruction(qc).probabilities()
    rows = []
    for index, prob in enumerate(probs):
        abc = "".join(str((index >> q) & 1) for q in range(3))
        used, value = allocation([int(c) for c in abc])
        rows.append((abc, float(prob), used, value, used <= CAPACITY))
    return sorted(rows, key=lambda r: r[1], reverse=True)

def decode_counts_key(key):
    # Qiskit displays classical bit 2 first; map display CBA -> application ABC.
    s = key.replace(" ", "")
    return s[-1] + s[-2] + s[-3]

def run_aer(gammas, betas, shots=4096):
    if not HAVE_AER:
        print("Aer unavailable; skipping shot-based run. Statevector results still work.")
        return None
    qc = qaoa_circuit(gammas, betas, measure=True)
    counts = Aer.get_backend("aer_simulator").run(qc, shots=shots).result().get_counts()
    print("\\nShot results (displayed CBA -> application ABC):")
    for key, n in sorted(counts.items(), key=lambda kv: kv[1], reverse=True):
        abc = decode_counts_key(key)
        used, value = allocation([int(c) for c in abc])
        print(f"{key} -> ABC={abc}, shots={n}, water={used}, benefit={value}, feasible={used <= CAPACITY}")
    return counts

def threshold_comparator_2bit(threshold=2, measure=True):
    """q0=W0, q1=W1, q2=output; computes output XOR [W >= threshold]."""
    if threshold not in range(4):
        raise ValueError("threshold must be 0, 1, 2, or 3")
    qc = QuantumCircuit(3, 1)
    for w in range(4):
        if w >= threshold:
            vals = [(w >> 0) & 1, (w >> 1) & 1]
            for q, bit in enumerate(vals):
                if bit == 0:
                    qc.x(q)
            qc.mcx([0, 1], 2)
            for q, bit in enumerate(vals):
                if bit == 0:
                    qc.x(q)
    if measure:
        qc.measure(2, 0)
    return qc

if __name__ == "__main__":
    print("QUANTUM-ASSISTED IRRIGATION WATER ALLOCATION")
    feasible = []
    for x in itertools.product([0, 1], repeat=3):
        used, value = allocation(x)
        if used <= CAPACITY:
            feasible.append((value, x, used))
    best = max(feasible)
    print(f"Classical optimum ABC={''.join(map(str, best[1]))}; water={best[2]}; benefit={best[0]}")

    print("\\nQUBO energies:")
    energies = []
    for x in itertools.product([0, 1], repeat=3):
        energies.append((qubo_energy(x), x))
        print(f"{x}: {qubo_energy(x):.1f}")
    print("QUBO minimum:", min(energies))
    validate_mapping()

    print("\\nOptimizing QAOA p=1 (multi-start COBYLA)...")
    res = optimize_qaoa()
    gammas, betas = res.x[:1], res.x[1:]
    print("Best expectation:", res.fun)
    print("gamma:", gammas, "beta:", betas)
    print("\\nQAOA circuit:")
    print(qaoa_circuit(gammas, betas, measure=True).draw("text"))
    print("\\nTop statevector probabilities (ABC order):")
    for abc, prob, used, value, ok in statevector_results(gammas, betas):
        print(f"ABC={abc}, probability={prob:.4f}, water={used}, benefit={value}, feasible={ok}")
    run_aer(gammas, betas)

    print("\\nComparator truth table W >= 2:")
    for w in range(4):
        print(f"W={w:02b} ({w}) -> {int(w >= 2)}")
    print("\\nComparator circuit:")
    print(threshold_comparator_2bit(2).draw("text"))
    print("\\nExpected classical application result: A+B (110), water=3, benefit=8.")
    print("QAOA is probabilistic; report actual probabilities, not guaranteed success or quantum speedup.")


QUANTUM-ASSISTED IRRIGATION WATER ALLOCATION
Classical optimum ABC=110; water=3; benefit=8
\nQUBO energies:
(0, 0, 0): 90.0
(0, 0, 1): -6.0
(0, 1, 0): 5.0
(0, 1, 1): 29.0
(1, 0, 0): 37.0
(1, 0, 1): 1.0
(1, 1, 0): -8.0
(1, 1, 1): 76.0
QUBO minimum: (-8.0, (1, 1, 0))
PASS: QUBO and Ising energies agree for all 8 bit strings.
\nOptimizing QAOA p=1 (multi-start COBYLA)...
Best expectation: 2.1066128209576567
gamma: [3.18546235] beta: [3.65672313]
\nQAOA circuit:
        ┌───┐┌────────────┐                          ┌────────────┐»
   q_0: ┤ H ├┤ Rz(9.5564) ├─■────────────■───────────┤ Rx(7.3134) ├»
        ├───┤├────────────┤ │ZZ(63.709)  │           └────────────┘»
   q_1: ┤ H ├┤ Rz(15.927) ├─■────────────┼────────────■────────────»
        ├───┤├────────────┤              │ZZ(95.564)  │ZZ(191.13)  »
   q_2: ┤ H ├┤ Rz(19.113) ├──────────────■────────────■────────────»
        └───┘└────────────┘                                        »
meas: 3/══════════════════════════════════════════════

In [5]:
        for q in range(3):
            qc.rz(2 * gamma * H_LINEAR[q], q)
        for (i, j), b in H_QUADRATIC.items():
            qc.rzz(2 * gamma * b, i, j)
        for q in range(3):
            qc.rx(2 * beta, q)
    if measure:
        qc.measure_all()
    return qc

def expectation(params, p=1):
    qc = qaoa_circuit(params[:p], params[p:])
    probs = Statevector.from_instruction(qc).probabilities()
    e = 0.0
    for index, prob in enumerate(probs):
        x = [(index >> q) & 1 for q in range(3)]  # q0=A, q1=B, q2=C
        e += prob * ising_energy(x)
    return float(np.real(e))

def optimize_qaoa(p=1, starts=10, maxiter=300, seed=7):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(starts):
        x0 = rng.uniform(0, 2*np.pi, 2*p)
        res = minimize(lambda v: expectation(v, p), x0, method="COBYLA",
                       options={"maxiter": maxiter, "rhobeg": 0.5})
        if best is None or res.fun < best.fun:
            best = res
    return best

def statevector_results(gammas, betas):
    qc = qaoa_circuit(gammas, betas)
    probs = Statevector.from_instruction(qc).probabilities()
    rows = []
    for index, prob in enumerate(probs):
        abc = "".join(str((index >> q) & 1) for q in range(3))
        used, value = allocation([int(c) for c in abc])
        rows.append((abc, float(prob), used, value, used <= CAPACITY))
    return sorted(rows, key=lambda r: r[1], reverse=True)

def decode_counts_key(key):
    # Qiskit displays classical bit 2 first; map display CBA -> application ABC.
    s = key.replace(" ", "")
    return s[-1] + s[-2] + s[-3]

def run_aer(gammas, betas, shots=4096):
    if not HAVE_AER:
        print("Aer unavailable; skipping shot-based run. Statevector results still work.")
        return None
    qc = qaoa_circuit(gammas, betas, measure=True)
    counts = Aer.get_backend("aer_simulator").run(qc, shots=shots).result().get_counts()
    print("\\nShot results (displayed CBA -> application ABC):")
    for key, n in sorted(counts.items(), key=lambda kv: kv[1], reverse=True):
        abc = decode_counts_key(key)
        used, value = allocation([int(c) for c in abc])
        print(f"{key} -> ABC={abc}, shots={n}, water={used}, benefit={value}, feasible={used <= CAPACITY}")
    return counts

def threshold_comparator_2bit(threshold=2, measure=True):
    """q0=W0, q1=W1, q2=output; computes output XOR [W >= threshold]."""
    if threshold not in range(4):
        raise ValueError("threshold must be 0, 1, 2, or 3")
    qc = QuantumCircuit(3, 1)
    for w in range(4):
        if w >= threshold:
            vals = [(w >> 0) & 1, (w >> 1) & 1]
            for q, bit in enumerate(vals):
                if bit == 0:
                    qc.x(q)
            qc.mcx([0, 1], 2)
            for q, bit in enumerate(vals):
                if bit == 0:
                    qc.x(q)
    if measure:
        qc.measure(2, 0)
    return qc

if __name__ == "__main__":
    print("QUANTUM-ASSISTED IRRIGATION WATER ALLOCATION")
    feasible = []
    for x in itertools.product([0, 1], repeat=3):
        used, value = allocation(x)
        if used <= CAPACITY:
            feasible.append((value, x, used))
    best = max(feasible)
    print(f"Classical optimum ABC={''.join(map(str, best[1]))}; water={best[2]}; benefit={best[0]}")

    print("\\nQUBO energies:")
    energies = []
    for x in itertools.product([0, 1], repeat=3):
        energies.append((qubo_energy(x), x))
        print(f"{x}: {qubo_energy(x):.1f}")
    print("QUBO minimum:", min(energies))
    validate_mapping()

    print("\\nOptimizing QAOA p=1 (multi-start COBYLA)...")
    res = optimize_qaoa()
    gammas, betas = res.x[:1], res.x[1:]
    print("Best expectation:", res.fun)
    print("gamma:", gammas, "beta:", betas)
    print("\\nQAOA circuit:")
    print(qaoa_circuit(gammas, betas, measure=True).draw("text"))
    print("\\nTop statevector probabilities (ABC order):")
    for abc, prob, used, value, ok in statevector_results(gammas, betas):
        print(f"ABC={abc}, probability={prob:.4f}, water={used}, benefit={value}, feasible={ok}")
    run_aer(gammas, betas)

    print("\\nComparator truth table W >= 2:")
    for w in range(4):
        print(f"W={w:02b} ({w}) -> {int(w >= 2)}")
    print("\\nComparator circuit:")
    print(threshold_comparator_2bit(2).draw("text"))
    print("\\nExpected classical application result: A+B (110), water=3, benefit=8.")
    print("QAOA is probabilistic; report actual probabilities, not guaranteed success or quantum speedup.")

IndentationError: expected an indented block after 'if' statement on line 7 (1448199386.py, line 8)

In [7]:

def qaoa_circuit(gammas, betas, measure=False):
    gammas = np.atleast_1d(gammas)
    betas = np.atleast_1d(betas)

    if len(gammas) != len(betas):
        raise ValueError("gamma and beta arrays must have equal lengths")

    qc = QuantumCircuit(3)
    qc.h(range(3))

    for gamma, beta in zip(gammas, betas):

        # Cost Hamiltonian: single-qubit Z terms
        for q in range(3):
            qc.rz(2 * gamma * H_LINEAR[q], q)

        # Cost Hamiltonian: two-qubit ZZ terms
        for (i, j), coeff in H_QUADRATIC.items():
            qc.rzz(2 * gamma * coeff, i, j)

        # Mixer Hamiltonian
        for q in range(3):
            qc.rx(2 * beta, q)

    # IMPORTANT: indent the measurement statement
    # inside the if block
    if measure:
        qc.measure_all()

    return qc


In [9]:

test_circuit = qaoa_circuit(
    gammas=[0.1],
    betas=[0.2],
    measure=True
)

print(test_circuit.draw("text"))


        ┌───┐┌─────────┐                ┌─────────┐            ░ ┌─┐      
   q_0: ┤ H ├┤ Rz(0.3) ├─■───────■──────┤ Rx(0.4) ├────────────░─┤M├──────
        ├───┤├─────────┤ │ZZ(2)  │      └─────────┘┌─────────┐ ░ └╥┘┌─┐   
   q_1: ┤ H ├┤ Rz(0.5) ├─■───────┼─────────■───────┤ Rx(0.4) ├─░──╫─┤M├───
        ├───┤├─────────┤         │ZZ(3)    │ZZ(6)  ├─────────┤ ░  ║ └╥┘┌─┐
   q_2: ┤ H ├┤ Rz(0.6) ├─────────■─────────■───────┤ Rx(0.4) ├─░──╫──╫─┤M├
        └───┘└─────────┘                           └─────────┘ ░  ║  ║ └╥┘
meas: 3/══════════════════════════════════════════════════════════╩══╩══╩═
                                                                  0  1  2 
